# Fig 3C — CS changes and SER fallback (data processing)

GTEx simulation: fraction of regions where the SuSiE-rRSS credible sets differ from SuSiE-RSS, and fraction that fall back to SER, per LD reference.

**Output:** `figure_data/Fig3C.CS_Change_SER_Fraction.rds` (+ `.csv`)

In [1]:
source("../../R/utils.R")

data_name <- "Fig3C.CS_Change_SER_Fraction"

res   <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_fine/summary_cs_similarity/cs_mix_vs_base_combined.rds")
susie <- readRDS("/hpc/users/sunh17/haochen/rss_qc/2026_output/Gtex_fine/summary/result_susie_bias_summary_summary.rds")

In [2]:
## CS agreement between SuSiE-rRSS (EB-mix) and SuSiE-RSS per reference
res %>%
  group_by(reference) %>%
  summarize(
    n_regions     = n(),
    same          = sum(same_all),                 # total same (incl. both-empty)
    pct_same      = round(mean(same_all) * 100, 1),
    both_empty    = sum(both_empty),
    same_nonempty = sum(same_all & !both_empty),    # same AND at least one CS found
    .groups = "drop"
  ) %>%
  arrange(match(reference, c("insample", "ADSP", "UKB", "kg")))

reference,n_regions,same,pct_same,both_empty,same_nonempty
<chr>,<int>,<int>,<dbl>,<int>,<int>
insample,7200,6849,95.1,997,0
ADSP,7200,6316,87.7,918,0
UKB,7200,6204,86.2,913,0
kg,7200,4331,60.2,884,0


In [3]:
## ---- per-region correction status from the EB-mix fits ----------------------
mix_methods <- c("insample", "ADSP", "UKB", "kg")

bias_tbl <- susie %>%
  filter(method %in% paste0(mix_methods, "_mix")) %>%        # exact _mix rows (not _mix_new / _SER)
  mutate(
    ref = str_remove(method, "_mix$"),
    status = case_when(
      R_reliability_flag %in% TRUE                 ~ "SER fallback",
      !is.na(lambda_bias) & lambda_bias != 0       ~ "Corrected",
      TRUE                                         ~ "No Correction"
    )
  )

In [4]:
## per-ref status counts joined with the CS-agreement summary (table for the text)
status_summ <- bias_tbl %>%
  group_by(ref) %>%
  summarize(
    n_bias        = n(),
    SER_fallback  = sum(status == "SER fallback"),
    Corrected     = sum(status == "Corrected"),
    No_Correction = sum(status == "No Correction"),
    pct_SER       = round(mean(status == "SER fallback") * 100, 1),
    .groups = "drop"
  )

cs_summ <- res %>%
  group_by(reference) %>%
  summarize(
    n_regions     = n(),
    same          = sum(same_all),
    same_nonempty = sum(same_all & !both_empty),
    pct_same      = round(mean(same_all) * 100, 1),
    .groups = "drop"
  )

combined <- cs_summ %>%
  left_join(status_summ, by = c("reference" = "ref")) %>%
  arrange(match(reference, c("insample", "ADSP", "UKB", "kg")))

combined

reference,n_regions,same,same_nonempty,pct_same,n_bias,SER_fallback,Corrected,No_Correction,pct_SER
<chr>,<int>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>,<dbl>
insample,7200,6849,5852,95.1,7200,1,972,6227,0.0
ADSP,7200,6316,5398,87.7,7200,71,2919,4210,1.0
UKB,7200,6204,5291,86.2,7200,275,2897,4028,3.8
kg,7200,4331,3447,60.2,7200,393,2558,4249,5.5


In [5]:
## ---- categorize per region -> plot-ready percentages ------------------------
ref_lab <- c(insample = "GTEx", ADSP = "ADSP", UKB = "UKB", kg = "1000G")

file_col <- grep("file", names(bias_tbl), value = TRUE)[1]   # finds out_file / input_file / file
message("using file column: ", file_col)

bias_key <- bias_tbl %>%
  transmute(file = basename(.data[[file_col]]), reference = ref, status)

plot_dt <- res %>%
  select(file, reference, same_all) %>%
  left_join(bias_key, by = c("file", "reference")) %>%
  mutate(category = case_when(
    status == "SER fallback" ~ "SER fallback",
    !same_all                ~ "Different",
    TRUE                     ~ "Same"
  )) %>%
  count(reference, category) %>%
  group_by(reference) %>%
  mutate(pct = n / sum(n) * 100) %>%          # denominator = ALL regions
  ungroup() %>%
  filter(category != "Same") %>%              # leave out the Same section
  mutate(
    reference = factor(ref_lab[reference], levels = unname(ref_lab)),
    category  = factor(category, levels = names(pal_cs_change))
  )

save_data(plot_dt, data_name)

using file column: input_file

